# H5 · `scripts/runstore.py`

## What this file is for

Every run, kept. One append-only JSON-lines file per month under `results/`; one line per run,
holding the configuration, the summary and every jurisdiction row. Nothing here updates or deletes a
line -- a corrected run is a new line, and the old one stays as the record of what was believed at
the time.

**It lives in `scripts/`, not `ui/`, and that is deliberate.** The store is a results concern, not
an interface one: the CLI writes to it, the browser writes to it, and `qa.py`'s live-call budget
counts from it. Putting it under `ui/` would force `sweep.py` to import the interface just to
record a run -- inverting the one-way dependency `ui -> scripts -> gl_engine` this project enforces.
`tests/verify_tester.py` asserts against exactly that inversion.

**This notebook does not write to the real store.** Every example below points `runstore.RESULTS`
at a temporary directory first and restores it after, because a notebook that appends a demo run to
`results/` every time someone executes it would corrupt the very thing this file exists to keep
honest.

**Depends on:** nothing in this set. Read by [`H2 qa.py`](02-qa.ipynb), [`H4 sweep.py`](04-sweep.ipynb)
and [`H6 charts.py`](06-charts.ipynb).

## Its public surface

Generated from the module, so it can't drift.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
sys.path.insert(0, str(Path.cwd().parent.parent / "scripts"))

import inspect
import runstore as store

for name, obj in vars(store).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != store.__name__:
        continue
    if inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

`append` writes one line; `runs()` reads them back newest first. Pointed at a temporary directory so
this cell is safe to re-run.

In [ ]:
import tempfile

real_results = store.RESULTS
store.RESULTS = Path(tempfile.mkdtemp())

line = store.append(
    {"fingerprint": "demo1", "describes": "the base risk, unvaried",
     "live_calls": 0, "rated": 1, "agree": 0, "total": 1},
    [{"juris": "TX", "status": "RATED", "ours": "8896"}],
    label="notebook demo")
print("stored id:", line["id"])
print("runs():", [(r["id"], r["label"]) for r in store.runs(limit=5)])

## The interesting case

### Coverage counts a `RATED` row and a `NOT APPLICABLE` row differently, on purpose

`coverage()` answers *"how narrow is the claim?"* -- and a `NOT APPLICABLE` row proves the
jurisdiction was asked and declined, which is worth knowing but is **not** coverage. Counting it as
coverage would let a control look tested in a state where it was never actually exercised.

In [ ]:
store.append(
    {"fingerprint": "demo2", "describes": "occurrence_limit=2,000,000 CSL",
     "config": {"occurrence_limit": "2,000,000 CSL"},
     "live_calls": 0, "rated": 1, "agree": 0, "total": 2},
    [{"juris": "TX", "status": "RATED", "ours": "8896"},
     {"juris": "AK", "status": "NOT APPLICABLE"}],
    label="qa T1 demo")

cov = store.coverage()
print("rated       :", cov["rated"])
print("declined    :", cov["declined"])

### The QA rollup is worst-first, never averaged

`qa_rollup` folds every row for a jurisdiction across many runs into one status, and when a
jurisdiction disagreed in even one of them, that is what shows -- however much else agreed. A tile
that averaged would hide the one scenario that matters behind the many that don't.

In [ ]:
r = store.qa_rollup("T1")
print("status :", r["status"])
print("counts :", r["counts"])

store.RESULTS = real_results               # done -- back to the real store
print()
print("real store untouched, runs still there:", len(store.runs(limit=3)))

## What it refuses

Nothing here raises -- a torn or hand-edited line is the one thing this module is defensive about,
and it refuses to pretend the line was never written.

In [ ]:
import tempfile as _tf

tmp = Path(_tf.mkdtemp())
(tmp / "runs-2026-01.jsonl").write_text(
    '{"v": 1, "id": "ok-line", "at": 1, "summary": {}, "rows": []}\n'
    'not json at all\n', encoding="utf-8")

real_results = store.RESULTS
store.RESULTS = tmp
bad = [r for r in store.runs(limit=10) if r.get("unreadable")]
store.RESULTS = real_results
print("unreadable lines surfaced, not dropped:", len(bad), bad[0]["id"] if bad else None)

## Try it yourself

1. `history(fingerprint)` is what the agreement-over-time chart draws. Why oldest-first, when
   `runs()` is newest-first -- what would a chart drawn the other way look like?
2. `defects()` reports first-seen and last-seen rather than a current-state boolean. What does a
   defect that was fixed and then reappeared look like in that shape, that a boolean would hide?
3. `LINE_VERSION` exists but nothing reads it yet. If the stored shape changed, what would a reader
   need to do differently for `v: 1` versus a hypothetical `v: 2`?

In [ ]:
# your turn